# Final BI Decision Support Dashboard - Notebook



Dataset: public UCI Online Retail transactions.


## 1. Install Analytics Dependencies

In [11]:
%pip install -q pandas numpy openpyxl plotly scikit-learn joblib

## 2. Imports and Output Folders

In [12]:
from dataclasses import dataclass
from pathlib import Path
from time import perf_counter
import json
import os
import platform
import shutil
import warnings
import zipfile

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
try:
    from IPython.display import display
except ImportError:
    display = print
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, silhouette_score
from sklearn.preprocessing import StandardScaler

WORK_DIR = Path.cwd()
RAW_DIR = WORK_DIR / "data" / "raw"
PROCESSED_DIR = WORK_DIR / "data" / "processed"
MODELS_DIR = WORK_DIR / "models"
EVIDENCE_DIR = WORK_DIR / "evidence"

for folder in [RAW_DIR, PROCESSED_DIR, MODELS_DIR, EVIDENCE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


def safe_divide(numerator, denominator, default=0.0):
    if denominator in (0, None) or pd.isna(denominator):
        return default
    return numerator / denominator


def short_text(value, maximum=45):
    text = str(value)
    return text if len(text) <= maximum else text[: maximum - 3] + "..."


print("Analytics workspace:", WORK_DIR)
print("Python version:", platform.python_version())

Analytics workspace: /content
Python version: 3.12.13


## 3. Upload or Locate the Dataset

Upload `Online Retail.xlsx`

In [13]:
def ensure_dataset_available():
    candidates = [
        RAW_DIR / "Online Retail.xlsx",
        WORK_DIR / "Online Retail.xlsx",
        Path("/content/Online Retail.xlsx"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            "Online Retail.xlsx was not found. Place it in the current folder or data/raw/."
        ) from exc

    uploaded = files.upload()
    excel_names = [name for name in uploaded if name.lower().endswith((".xlsx", ".xls"))]
    if not excel_names:
        raise ValueError("Upload the UCI Online Retail Excel workbook.")
    source = Path(excel_names[0])
    target = RAW_DIR / "Online Retail.xlsx"
    shutil.copy2(source, target)
    return target


DATASET_PATH = ensure_dataset_available()
print("Dataset selected:", DATASET_PATH)

Dataset selected: /content/Online Retail.xlsx


## 4. Final Data Loading and Preprocessing Functions

In [14]:
REQUIRED_COLUMNS = [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country",
]

NON_MERCHANDISE_CODES = {
    "AMAZONFEE",
    "BANK CHARGES",
    "C2",
    "CRUK",
    "D",
    "DOT",
    "M",
    "PADS",
    "POST",
    "S",
}


def validate_columns(df: pd.DataFrame) -> None:
    """Raise a clear error when required dataset columns are missing."""

    missing = [column for column in REQUIRED_COLUMNS if column not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")


def _clean_identifier(value) -> object:
    """Convert Excel numeric IDs into readable strings without .0 suffixes."""

    if pd.isna(value):
        return pd.NA
    try:
        numeric = float(value)
        if np.isfinite(numeric) and numeric.is_integer():
            return str(int(numeric))
    except (TypeError, ValueError):
        pass
    text = str(value).strip()
    return text if text else pd.NA


def standardize_raw_data(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize column names, types, and basic text fields."""

    standardized = df.copy()
    standardized.columns = [str(column).strip() for column in standardized.columns]
    validate_columns(standardized)

    standardized["InvoiceNo"] = standardized["InvoiceNo"].apply(_clean_identifier)
    standardized["StockCode"] = standardized["StockCode"].apply(_clean_identifier)
    standardized["CustomerID"] = standardized["CustomerID"].apply(_clean_identifier)
    standardized["Description"] = (
        standardized["Description"]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    standardized["Country"] = standardized["Country"].astype("string").str.strip()
    standardized["Quantity"] = pd.to_numeric(standardized["Quantity"], errors="coerce")
    standardized["UnitPrice"] = pd.to_numeric(standardized["UnitPrice"], errors="coerce")
    standardized["InvoiceDate"] = pd.to_datetime(standardized["InvoiceDate"], errors="coerce")

    return standardized


def prepare_retail_data(raw_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Create analysis-ready dataframes.

    Returns all standardized rows plus a clean positive-sales table. Cancelled
    and invalid rows are preserved in the first dataframe for data-quality and
    cancellation analysis.
    """

    prepared = standardize_raw_data(raw_df)

    prepared["IsCancelled"] = (
        prepared["InvoiceNo"].astype("string").str.startswith("C", na=False)
        | (prepared["Quantity"] < 0)
    )
    prepared["IsMissingCustomer"] = prepared["CustomerID"].isna()
    prepared["IsZeroOrNegativePrice"] = prepared["UnitPrice"].fillna(0) <= 0
    prepared["IsZeroOrNegativeQuantity"] = prepared["Quantity"].fillna(0) <= 0
    prepared["IsNonMerchandise"] = (
        prepared["StockCode"].astype("string").str.upper().isin(NON_MERCHANDISE_CODES)
        | prepared["Description"].astype("string").str.upper().str.contains(
            r"POSTAGE|BANK CHARGE|AMAZON FEE|MANUAL|DISCOUNT|CARRIAGE",
            regex=True,
            na=False,
        )
    )
    prepared["Revenue"] = prepared["Quantity"] * prepared["UnitPrice"]
    prepared["InvoiceDateOnly"] = prepared["InvoiceDate"].dt.date
    prepared["InvoiceDateOnly"] = pd.to_datetime(prepared["InvoiceDateOnly"])
    prepared["Month"] = prepared["InvoiceDate"].dt.to_period("M").dt.to_timestamp()
    prepared["Year"] = prepared["InvoiceDate"].dt.year
    prepared["YearMonth"] = prepared["InvoiceDate"].dt.to_period("M").astype("string")

    valid_sales_mask = (
        prepared["InvoiceDate"].notna()
        & prepared["Description"].notna()
        & prepared["Quantity"].gt(0)
        & prepared["UnitPrice"].gt(0)
        & ~prepared["IsCancelled"]
    )
    prepared["IsValidSale"] = valid_sales_mask
    clean_sales = prepared.loc[valid_sales_mask].copy()

    return prepared, clean_sales


def summarize_data_quality(prepared_df: pd.DataFrame) -> dict:
    """Summarize records that need cleaning or special handling."""

    return {
        "total_rows": int(len(prepared_df)),
        "missing_customer_rows": int(prepared_df["CustomerID"].isna().sum()),
        "cancelled_or_return_rows": int(prepared_df["IsCancelled"].sum()),
        "zero_or_negative_price_rows": int(prepared_df["IsZeroOrNegativePrice"].sum()),
        "zero_or_negative_quantity_rows": int(prepared_df["IsZeroOrNegativeQuantity"].sum()),
        "valid_sales_rows": int(prepared_df["IsValidSale"].sum()),
        "non_merchandise_sales_rows": int(
            (prepared_df["IsValidSale"] & prepared_df["IsNonMerchandise"]).sum()
        ),
    }


def save_processed_data(clean_sales: pd.DataFrame, output_path: str | Path) -> Path:
    """Save cleaned sales data as CSV for faster app loading."""

    path = Path(output_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    compression = "gzip" if path.suffix.lower() == ".gz" else None
    clean_sales.to_csv(path, index=False, compression=compression)
    return path


def save_prepared_data(prepared_df: pd.DataFrame, output_path: str | Path) -> Path:
    """Save all standardized rows, including quality flags, as compressed CSV."""

    return save_processed_data(prepared_df, output_path)

## 5. Final KPI, Sales, Customer, and Product Functions

In [15]:
def calculate_overview_kpis(clean_sales: pd.DataFrame) -> dict:
    """Calculate high-level executive KPIs."""

    total_revenue = float(clean_sales["Revenue"].sum())
    total_orders = int(clean_sales["InvoiceNo"].nunique())
    total_customers = int(clean_sales["CustomerID"].nunique(dropna=True))
    total_quantity = float(clean_sales["Quantity"].sum())

    return {
        "total_revenue": total_revenue,
        "total_orders": total_orders,
        "total_customers": total_customers,
        "average_order_value": safe_divide(total_revenue, total_orders),
        "total_quantity": total_quantity,
        "unique_products": int(clean_sales["StockCode"].nunique()),
        "countries": int(clean_sales["Country"].nunique()),
        "date_min": clean_sales["InvoiceDate"].min(),
        "date_max": clean_sales["InvoiceDate"].max(),
    }


def monthly_sales(clean_sales: pd.DataFrame) -> pd.DataFrame:
    """Aggregate sales by month."""

    grouped = (
        clean_sales.groupby("Month", as_index=False)
        .agg(
            Revenue=("Revenue", "sum"),
            Quantity=("Quantity", "sum"),
            Orders=("InvoiceNo", "nunique"),
            Customers=("CustomerID", "nunique"),
        )
        .sort_values("Month")
    )
    grouped["AverageOrderValue"] = grouped["Revenue"] / grouped["Orders"].replace(0, pd.NA)
    grouped["IsCompleteMonth"] = True
    if not grouped.empty:
        minimum_date = pd.to_datetime(clean_sales["InvoiceDate"].min())
        maximum_date = pd.to_datetime(clean_sales["InvoiceDate"].max())
        first_month = minimum_date.to_period("M").to_timestamp()
        last_month = maximum_date.to_period("M").to_timestamp()
        if minimum_date.normalize() > first_month:
            grouped.loc[grouped["Month"].eq(first_month), "IsCompleteMonth"] = False
        if maximum_date.normalize() < (last_month + pd.offsets.MonthEnd(0)).normalize():
            grouped.loc[grouped["Month"].eq(last_month), "IsCompleteMonth"] = False
    return grouped


def complete_monthly_sales(clean_sales: pd.DataFrame) -> pd.DataFrame:
    """Return monthly aggregates that cover complete calendar months only."""

    monthly = monthly_sales(clean_sales)
    return monthly.loc[monthly["IsCompleteMonth"]].copy()


def sales_by_country(clean_sales: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    """Rank countries or markets by revenue."""

    return (
        clean_sales.groupby("Country", as_index=False)
        .agg(
            Revenue=("Revenue", "sum"),
            Quantity=("Quantity", "sum"),
            Orders=("InvoiceNo", "nunique"),
            Customers=("CustomerID", "nunique"),
        )
        .sort_values("Revenue", ascending=False)
        .head(top_n)
    )


def top_products(
    clean_sales: pd.DataFrame,
    top_n: int = 10,
    include_non_merchandise: bool = False,
) -> pd.DataFrame:
    """Rank products by revenue."""

    product_sales = clean_sales.copy()
    if not include_non_merchandise and "IsNonMerchandise" in product_sales.columns:
        product_sales = product_sales.loc[~product_sales["IsNonMerchandise"]].copy()

    grouped = (
        product_sales.groupby(["StockCode", "Description"], as_index=False)
        .agg(
            Revenue=("Revenue", "sum"),
            Quantity=("Quantity", "sum"),
            Orders=("InvoiceNo", "nunique"),
            Customers=("CustomerID", "nunique"),
            AvgUnitPrice=("UnitPrice", "mean"),
        )
        .sort_values("Revenue", ascending=False)
        .head(top_n)
    )
    return grouped


def product_performance(
    clean_sales: pd.DataFrame,
    include_non_merchandise: bool = False,
) -> pd.DataFrame:
    """Create a product-level performance table."""

    product_sales = clean_sales.copy()
    if not include_non_merchandise and "IsNonMerchandise" in product_sales.columns:
        product_sales = product_sales.loc[~product_sales["IsNonMerchandise"]].copy()

    product_df = (
        product_sales.groupby(["StockCode", "Description"], as_index=False)
        .agg(
            Revenue=("Revenue", "sum"),
            Quantity=("Quantity", "sum"),
            Orders=("InvoiceNo", "nunique"),
            Customers=("CustomerID", "nunique"),
            AvgUnitPrice=("UnitPrice", "mean"),
            FirstSale=("InvoiceDate", "min"),
            LastSale=("InvoiceDate", "max"),
        )
        .sort_values("Revenue", ascending=False)
    )
    product_df["RevenuePerOrder"] = product_df["Revenue"] / product_df["Orders"].replace(0, pd.NA)
    return product_df


def classify_product_actions(product_df: pd.DataFrame) -> pd.DataFrame:
    """Assign transparent action categories using median revenue and demand."""

    classified = product_df.copy()
    if classified.empty:
        classified["ActionCategory"] = pd.Series(dtype="string")
        return classified

    revenue_median = classified["Revenue"].median()
    quantity_median = classified["Quantity"].median()
    high_revenue = classified["Revenue"] >= revenue_median
    high_quantity = classified["Quantity"] >= quantity_median
    classified["ActionCategory"] = "Review or rationalize"
    classified.loc[high_revenue & high_quantity, "ActionCategory"] = "Protect and promote"
    classified.loc[high_revenue & ~high_quantity, "ActionCategory"] = "Premium-value opportunity"
    classified.loc[~high_revenue & high_quantity, "ActionCategory"] = "Volume / margin review"
    return classified


def customer_value_table(clean_sales: pd.DataFrame) -> pd.DataFrame:
    """Create a customer-level sales summary."""

    customer_sales = clean_sales.dropna(subset=["CustomerID"]).copy()
    grouped = (
        customer_sales.groupby("CustomerID", as_index=False)
        .agg(
            Revenue=("Revenue", "sum"),
            Orders=("InvoiceNo", "nunique"),
            Quantity=("Quantity", "sum"),
            FirstPurchase=("InvoiceDate", "min"),
            LastPurchase=("InvoiceDate", "max"),
            Countries=("Country", "nunique"),
        )
        .sort_values("Revenue", ascending=False)
    )
    grouped["AverageOrderValue"] = grouped["Revenue"] / grouped["Orders"].replace(0, pd.NA)
    return grouped


def cancellation_summary(prepared_all: pd.DataFrame) -> dict:
    """Summarize cancelled/negative transactions."""

    cancelled = prepared_all[prepared_all["IsCancelled"]].copy()
    return {
        "cancelled_rows": int(len(cancelled)),
        "cancelled_invoices": int(cancelled["InvoiceNo"].nunique()) if not cancelled.empty else 0,
        "cancelled_quantity": float(cancelled["Quantity"].sum()) if not cancelled.empty else 0.0,
        "cancelled_value": float(cancelled["Revenue"].sum()) if not cancelled.empty else 0.0,
        "cancelled_share_rows": safe_divide(len(cancelled), len(prepared_all)),
    }


def slow_moving_products(clean_sales: pd.DataFrame, top_n: int = 15) -> pd.DataFrame:
    """Identify products with low quantity and older last-sale dates."""

    product_df = product_performance(clean_sales, include_non_merchandise=False)
    if product_df.empty:
        return product_df

    revenue_cutoff = product_df["Revenue"].quantile(0.25)
    quantity_cutoff = product_df["Quantity"].quantile(0.25)
    slow = product_df[
        (product_df["Revenue"] <= revenue_cutoff)
        & (product_df["Quantity"] <= quantity_cutoff)
    ].sort_values(["LastSale", "Revenue"], ascending=[True, True])
    return slow.head(top_n)

## 6. Final RFM and K-Means Segmentation Functions

In [16]:
@dataclass
class SegmentationResult:
    rfm: pd.DataFrame
    summary: pd.DataFrame
    metrics: dict
    model: KMeans
    scaler: StandardScaler


def build_rfm_table(clean_sales: pd.DataFrame, analysis_date: pd.Timestamp | None = None) -> pd.DataFrame:
    """Build a customer-level RFM table."""

    customer_sales = clean_sales.dropna(subset=["CustomerID"]).copy()
    if customer_sales.empty:
        return pd.DataFrame()

    max_date = customer_sales["InvoiceDate"].max()
    analysis_date = pd.to_datetime(analysis_date or max_date + pd.Timedelta(days=1))

    rfm = (
        customer_sales.groupby("CustomerID", as_index=False)
        .agg(
            LastPurchase=("InvoiceDate", "max"),
            FirstPurchase=("InvoiceDate", "min"),
            Frequency=("InvoiceNo", "nunique"),
            Monetary=("Revenue", "sum"),
            Quantity=("Quantity", "sum"),
        )
    )
    rfm["Recency"] = (analysis_date - rfm["LastPurchase"]).dt.days
    rfm["CustomerAgeDays"] = (analysis_date - rfm["FirstPurchase"]).dt.days.clip(lower=1)
    rfm["AverageOrderValue"] = rfm["Monetary"] / rfm["Frequency"].replace(0, pd.NA)
    rfm["PurchaseRatePerMonth"] = rfm["Frequency"] / (rfm["CustomerAgeDays"] / 30.4375)
    rfm["AnnualizedValueIndicator"] = (
        rfm["AverageOrderValue"] * rfm["Frequency"] * 365 / rfm["CustomerAgeDays"]
    )
    # This is a behavioural proxy, not a probabilistic customer lifetime value model.
    rfm["CLVIndicator"] = rfm["AnnualizedValueIndicator"]

    rfm = add_rfm_scores(rfm)
    return rfm.sort_values("Monetary", ascending=False)


def _score_series(series: pd.Series, high_is_good: bool = True) -> pd.Series:
    """Score a series from 1 to 5 using ranks so duplicate values are safe."""

    if series.nunique(dropna=True) <= 1:
        return pd.Series([3] * len(series), index=series.index)
    ranked = series.rank(method="first", ascending=high_is_good)
    scored = pd.qcut(ranked, q=5, labels=[1, 2, 3, 4, 5])
    return scored.astype(int)


def add_rfm_scores(rfm: pd.DataFrame) -> pd.DataFrame:
    """Add R, F, M, and combined RFM scores."""

    scored = rfm.copy()
    scored["RScore"] = _score_series(scored["Recency"], high_is_good=False)
    scored["FScore"] = _score_series(scored["Frequency"], high_is_good=True)
    scored["MScore"] = _score_series(scored["Monetary"], high_is_good=True)
    scored["RFMScore"] = scored["RScore"] + scored["FScore"] + scored["MScore"]
    return scored


def _cluster_label_map(summary: pd.DataFrame) -> dict:
    """Assign human-readable labels to cluster IDs based on business value."""

    ranked = summary.copy()
    ranked["RankScore"] = (
        ranked["Monetary"].rank(ascending=False)
        + ranked["Frequency"].rank(ascending=False)
        + ranked["Recency"].rank(ascending=True)
    )
    ranked = ranked.sort_values("RankScore")

    label_sets = {
        2: ["Champions", "At Risk"],
        3: ["Champions", "Loyal Customers", "At Risk"],
        4: ["Champions", "Loyal Customers", "Potential Loyalists", "At Risk"],
        5: ["Champions", "Loyal Customers", "Potential Loyalists", "At Risk", "Low Value"],
        6: [
            "Champions",
            "Loyal Customers",
            "Potential Loyalists",
            "New or Occasional",
            "At Risk",
            "Low Value",
        ],
    }
    label_pool = label_sets.get(len(ranked), [f"Segment {index + 1}" for index in range(len(ranked))])
    return {
        int(cluster_id): label_pool[index] if index < len(label_pool) else f"Segment {index + 1}"
        for index, cluster_id in enumerate(ranked["Cluster"].tolist())
    }


def segment_customers(rfm: pd.DataFrame, n_clusters: int = 4, random_state: int = 42) -> SegmentationResult:
    """Cluster customers using scaled log-transformed RFM features."""

    if rfm.empty:
        raise ValueError("RFM table is empty. Customer segmentation requires valid CustomerID values.")

    if len(rfm) < 2:
        raise ValueError("Customer segmentation requires at least two customers.")

    usable_clusters = min(max(2, n_clusters), len(rfm))
    features = rfm[["Recency", "Frequency", "Monetary"]].copy()
    features["Frequency"] = np.log1p(features["Frequency"])
    features["Monetary"] = np.log1p(features["Monetary"].clip(lower=0))

    scaler = StandardScaler()
    scaled = scaler.fit_transform(features)

    model = KMeans(n_clusters=usable_clusters, random_state=random_state, n_init=10)
    clusters = model.fit_predict(scaled)

    segmented = rfm.copy()
    segmented["Cluster"] = clusters

    summary = (
        segmented.groupby("Cluster", as_index=False)
        .agg(
            Customers=("CustomerID", "count"),
            Recency=("Recency", "mean"),
            Frequency=("Frequency", "mean"),
            Monetary=("Monetary", "mean"),
            TotalRevenue=("Monetary", "sum"),
            AvgOrderValue=("AverageOrderValue", "mean"),
            AvgRFMScore=("RFMScore", "mean"),
        )
        .sort_values("TotalRevenue", ascending=False)
    )
    labels = _cluster_label_map(summary)
    segmented["SegmentName"] = segmented["Cluster"].map(labels)
    summary["SegmentName"] = summary["Cluster"].map(labels)

    metrics = {
        "n_clusters": usable_clusters,
        "customers_segmented": int(len(segmented)),
        "silhouette_score": None,
    }
    if usable_clusters > 1 and len(segmented) > usable_clusters:
        metrics["silhouette_score"] = float(silhouette_score(scaled, clusters))

    return SegmentationResult(
        rfm=segmented.sort_values("Monetary", ascending=False),
        summary=summary,
        metrics=metrics,
        model=model,
        scaler=scaler,
    )


def evaluate_cluster_counts(
    rfm: pd.DataFrame,
    minimum: int = 2,
    maximum: int = 6,
    random_state: int = 42,
) -> pd.DataFrame:
    """Compare candidate cluster counts using silhouette score and inertia."""

    if len(rfm) < 3:
        return pd.DataFrame(columns=["Clusters", "SilhouetteScore", "Inertia"])

    upper = min(maximum, len(rfm) - 1)
    rows = []
    for clusters in range(minimum, upper + 1):
        result = segment_customers(rfm, n_clusters=clusters, random_state=random_state)
        rows.append(
            {
                "Clusters": clusters,
                "SilhouetteScore": result.metrics["silhouette_score"],
                "Inertia": float(result.model.inertia_),
            }
        )
    return pd.DataFrame(rows)


def save_segmentation_result(result: SegmentationResult, output_dir: str | Path) -> None:
    """Save segmentation tables and model artifacts."""

    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    result.rfm.to_csv(output / "rfm_segments.csv", index=False)
    result.summary.to_csv(output / "segment_summary.csv", index=False)
    joblib.dump({"model": result.model, "scaler": result.scaler, "metrics": result.metrics}, output / "customer_segmentation.joblib")

## 7. Final Forecasting and Evaluation Functions

In [17]:
@dataclass
class ForecastResult:
    monthly: pd.DataFrame
    model_monthly: pd.DataFrame
    comparison: pd.DataFrame
    backtest: pd.DataFrame
    future_forecast: pd.DataFrame
    best_model: str
    excluded_partial_months: int


def aggregate_monthly(
    clean_sales: pd.DataFrame,
    target: str = "Revenue",
    complete_months_only: bool = False,
) -> pd.DataFrame:
    """Aggregate revenue or quantity by month."""

    if target not in {"Revenue", "Quantity"}:
        raise ValueError("target must be either 'Revenue' or 'Quantity'")

    monthly = monthly_sales(clean_sales)
    if complete_months_only:
        monthly = monthly.loc[monthly["IsCompleteMonth"]].copy()
    monthly["Target"] = monthly[target]
    return monthly


def _moving_average_forecast(history: list[float], horizon: int, window: int = 3) -> np.ndarray:
    """Forecast by repeatedly averaging the latest observations."""

    values = list(history)
    preds = []
    for _ in range(horizon):
        window_values = values[-window:] if len(values) >= window else values
        pred = float(np.mean(window_values))
        preds.append(pred)
        values.append(pred)
    return np.array(preds)


def _linear_trend_forecast(history: np.ndarray, horizon: int) -> np.ndarray:
    """Forecast with a simple linear trend over time index."""

    x_train = np.arange(len(history)).reshape(-1, 1)
    model = LinearRegression()
    model.fit(x_train, history)
    x_future = np.arange(len(history), len(history) + horizon).reshape(-1, 1)
    return model.predict(x_future)


def _naive_forecast(history: np.ndarray, horizon: int) -> np.ndarray:
    """Forecast by repeating the latest actual value."""

    return np.repeat(history[-1], horizon)


def _mape(actual: np.ndarray, predicted: np.ndarray) -> float:
    mask = actual != 0
    if not mask.any():
        return float("nan")
    return float(np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100)


def _metrics(actual: np.ndarray, predicted: np.ndarray) -> dict:
    return {
        "MAE": float(mean_absolute_error(actual, predicted)),
        "RMSE": float(np.sqrt(mean_squared_error(actual, predicted))),
        "MAPE": _mape(actual, predicted),
    }


def evaluate_forecast_models(
    monthly: pd.DataFrame,
    test_size: int = 3,
    return_backtest: bool = False,
) -> pd.DataFrame | tuple[pd.DataFrame, pd.DataFrame]:
    """Compare models using a chronological holdout period."""

    if len(monthly) < 6:
        raise ValueError("At least 6 monthly observations are recommended for forecast comparison.")

    test_size = min(test_size, max(1, len(monthly) // 4))
    train = monthly["Target"].iloc[:-test_size].astype(float).to_numpy()
    test = monthly["Target"].iloc[-test_size:].astype(float).to_numpy()

    predictions = {
        "Naive Last Value": _naive_forecast(train, len(test)),
        "Moving Average (3 months)": _moving_average_forecast(train.tolist(), len(test), window=3),
        "Linear Trend": _linear_trend_forecast(train, len(test)),
    }

    rows = []
    backtest_rows = []
    for name, pred in predictions.items():
        values = _metrics(test, pred)
        rows.append({"Model": name, **values})
        for month, actual, predicted in zip(monthly["Month"].iloc[-test_size:], test, pred):
            backtest_rows.append(
                {
                    "Month": month,
                    "Model": name,
                    "Actual": float(actual),
                    "Predicted": float(predicted),
                    "AbsoluteError": float(abs(actual - predicted)),
                }
            )
    comparison = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True)
    backtest = pd.DataFrame(backtest_rows)
    return (comparison, backtest) if return_backtest else comparison


def forecast_future(
    monthly: pd.DataFrame,
    model_name: str,
    periods: int = 3,
    forecast_after: pd.Timestamp | None = None,
) -> pd.DataFrame:
    """Forecast future months using the selected model."""

    history = monthly["Target"].astype(float).to_numpy()
    last_model_month = pd.to_datetime(monthly["Month"].max())
    forecast_after = pd.to_datetime(forecast_after or last_model_month)
    month_gap = max(
        0,
        (forecast_after.year - last_model_month.year) * 12
        + forecast_after.month
        - last_model_month.month,
    )
    full_horizon = periods + month_gap

    if model_name == "Moving Average (3 months)":
        all_predictions = _moving_average_forecast(history.tolist(), full_horizon, window=3)
    elif model_name == "Linear Trend":
        all_predictions = _linear_trend_forecast(history, full_horizon)
    else:
        all_predictions = _naive_forecast(history, full_horizon)

    pred = all_predictions[month_gap:]

    future_months = pd.date_range(forecast_after + pd.offsets.MonthBegin(1), periods=periods, freq="MS")
    return pd.DataFrame({"Month": future_months, "Forecast": pred.clip(min=0)})


def run_forecasting(clean_sales: pd.DataFrame, target: str = "Revenue", periods: int = 3, test_size: int = 3) -> ForecastResult:
    """Run the full forecasting workflow."""

    monthly = aggregate_monthly(clean_sales, target=target, complete_months_only=False)
    model_monthly = monthly.loc[monthly["IsCompleteMonth"]].copy()
    comparison, backtest = evaluate_forecast_models(
        model_monthly,
        test_size=test_size,
        return_backtest=True,
    )
    best_model = str(comparison.iloc[0]["Model"])
    future = forecast_future(
        model_monthly,
        best_model,
        periods=periods,
        forecast_after=monthly["Month"].max(),
    )
    return ForecastResult(
        monthly=monthly,
        model_monthly=model_monthly,
        comparison=comparison,
        backtest=backtest,
        future_forecast=future,
        best_model=best_model,
        excluded_partial_months=int((~monthly["IsCompleteMonth"]).sum()),
    )

## 8. Final Decision-Support Functions

In [18]:
def _insight(category: str, severity: str, title: str, message: str, recommendation: str) -> dict:
    return {
        "category": category,
        "severity": severity,
        "title": title,
        "message": message,
        "recommendation": recommendation,
    }


def generate_business_insights(clean_sales: pd.DataFrame, rfm_df: pd.DataFrame | None = None) -> list[dict]:
    """Generate simple business recommendations from dashboard outputs."""

    insights: list[dict] = []
    monthly = complete_monthly_sales(clean_sales)

    if len(monthly) >= 2:
        latest = monthly.iloc[-1]
        previous = monthly.iloc[-2]
        change = (latest["Revenue"] - previous["Revenue"]) / previous["Revenue"] if previous["Revenue"] else 0
        if change < -0.1:
            insights.append(
                _insight(
                    "Sales",
                    "High",
                    "Recent revenue decline detected",
                    f"Latest monthly revenue is {change:.1%} lower than the previous month.",
                    "Review recent product demand, market performance, and customer activity before planning promotions.",
                )
            )
        elif change > 0.1:
            insights.append(
                _insight(
                    "Sales",
                    "Positive",
                    "Recent revenue growth detected",
                    f"Latest monthly revenue is {change:.1%} higher than the previous month.",
                    "Identify the products and countries driving the increase and protect stock availability.",
                )
            )

    products = product_performance(clean_sales, include_non_merchandise=False)
    if not products.empty:
        top_product = products.iloc[0]
        insights.append(
            _insight(
                "Product",
                "Positive",
                "Top product deserves priority",
                f"{top_product['Description']} is the highest-revenue product in the dataset.",
                "Prioritize availability, monitor demand, and consider promotion around this product category.",
            )
        )

        slow = slow_moving_products(clean_sales, top_n=1)
        if not slow.empty:
            slow_product = slow.iloc[0]
            insights.append(
                _insight(
                    "Product",
                    "Medium",
                    "Slow-moving product review needed",
                    f"{slow_product['Description']} has low revenue and quantity movement.",
                    "Review whether this item needs bundling, discounting, repositioning, or removal.",
                )
            )

    if rfm_df is not None and not rfm_df.empty and "SegmentName" in rfm_df.columns:
        segment_revenue = rfm_df.groupby("SegmentName")["Monetary"].sum().sort_values(ascending=False)
        top_segment = segment_revenue.index[0]
        insights.append(
            _insight(
                "Customer",
                "Positive",
                "High-value customer segment identified",
                f"The {top_segment} segment contributes the largest customer revenue share.",
                "Use loyalty, retention, or targeted communication strategies for this segment.",
            )
        )

        at_risk = rfm_df[rfm_df["Recency"] > rfm_df["Recency"].quantile(0.75)]
        if not at_risk.empty:
            insights.append(
                _insight(
                    "Customer",
                    "Medium",
                    "Inactive customer group requires attention",
                    f"{len(at_risk):,} customers have relatively high recency values.",
                    "Create a re-engagement campaign for customers who have not purchased recently.",
                )
            )

    if not insights:
        insights.append(
            _insight(
                "General",
                "Info",
                "No major warning detected",
                "The current filtered dataset does not trigger a major alert.",
                "Continue monitoring KPIs, product movement, and customer segment changes.",
            )
        )

    return insights

## 9. Visualization Functions

In [19]:
COLOR_SEQUENCE = ["#111111", "#FF6B35", "#7F7F7F", "#B8BCC4", "#444444"]


def revenue_trend(monthly_df: pd.DataFrame, value_col: str = "Revenue") -> go.Figure:
    if "IsCompleteMonth" not in monthly_df.columns:
        fig = px.line(monthly_df, x="Month", y=value_col, markers=True, title=f"Monthly {value_col} Trend")
        fig.update_traces(line_color="#E85D2A", line_width=3)
        fig.update_layout(template="plotly_white", hovermode="x unified")
        return fig

    complete = monthly_df.loc[monthly_df["IsCompleteMonth"]].copy()
    partial = monthly_df.loc[~monthly_df["IsCompleteMonth"]].copy()
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=complete["Month"],
            y=complete[value_col],
            mode="lines+markers",
            name="Complete month",
            line=dict(color="#E85D2A", width=3),
        )
    )
    if not partial.empty:
        fig.add_trace(
            go.Scatter(
                x=partial["Month"],
                y=partial[value_col],
                mode="markers",
                name="Partial month",
                marker=dict(color="#D92D20", size=11, symbol="diamond"),
            )
        )
    fig.update_layout(title=f"Monthly {value_col} Trend")
    fig.update_layout(template="plotly_white", hovermode="x unified")
    return fig


def country_bar(country_df: pd.DataFrame) -> go.Figure:
    fig = px.bar(
        country_df.sort_values("Revenue"),
        x="Revenue",
        y="Country",
        orientation="h",
        title="Top Countries / Markets by Revenue",
        color_discrete_sequence=["#111111"],
    )
    fig.update_layout(template="plotly_white", yaxis_title="", xaxis_title="Revenue")
    return fig


def product_bar(product_df: pd.DataFrame, value_col: str = "Revenue", title: str = "Top Products") -> go.Figure:
    data = product_df.copy()
    data["Product"] = data["Description"].apply(short_text)
    fig = px.bar(
        data.sort_values(value_col),
        x=value_col,
        y="Product",
        orientation="h",
        title=title,
        color_discrete_sequence=["#111111"],
    )
    fig.update_layout(template="plotly_white", yaxis_title="", xaxis_title=value_col)
    return fig


def customer_scatter(customer_df: pd.DataFrame) -> go.Figure:
    fig = px.scatter(
        customer_df,
        x="Orders",
        y="Revenue",
        size="Quantity",
        hover_name="CustomerID",
        title="Customer Revenue vs Purchase Frequency",
        color_discrete_sequence=["#FF6B35"],
    )
    fig.update_layout(template="plotly_white")
    return fig


def rfm_segment_scatter(rfm_df: pd.DataFrame) -> go.Figure:
    fig = px.scatter(
        rfm_df,
        x="Frequency",
        y="Monetary",
        color="SegmentName",
        size="RFMScore",
        hover_name="CustomerID",
        title="Customer Segments by Frequency and Monetary Value",
        color_discrete_sequence=COLOR_SEQUENCE,
    )
    fig.update_layout(template="plotly_white")
    return fig


def segment_bar(summary_df: pd.DataFrame) -> go.Figure:
    fig = px.bar(
        summary_df.sort_values("TotalRevenue"),
        x="TotalRevenue",
        y="SegmentName",
        orientation="h",
        title="Customer Segment Revenue Contribution",
        color="Customers",
        color_continuous_scale=["#EDEDED", "#FF6B35"],
    )
    fig.update_layout(template="plotly_white", yaxis_title="", xaxis_title="Total Revenue")
    return fig


def forecast_chart(monthly_df: pd.DataFrame, future_df: pd.DataFrame) -> go.Figure:
    fig = go.Figure()
    complete = monthly_df.loc[monthly_df.get("IsCompleteMonth", True)].copy()
    partial = monthly_df.loc[~monthly_df.get("IsCompleteMonth", True)].copy()
    fig.add_trace(
        go.Scatter(
            x=complete["Month"],
            y=complete["Target"],
            mode="lines+markers",
            name="Actual - complete month",
            line=dict(color="#111111", width=3),
        )
    )
    if not partial.empty:
        fig.add_trace(
            go.Scatter(
                x=partial["Month"],
                y=partial["Target"],
                mode="markers",
                name="Actual - partial month",
                marker=dict(color="#D92D20", size=11, symbol="diamond"),
            )
        )
    fig.add_trace(
        go.Scatter(
            x=future_df["Month"],
            y=future_df["Forecast"],
            mode="lines+markers",
            name="Forecast",
            line=dict(color="#FF6B35", width=3, dash="dash"),
        )
    )
    fig.update_layout(template="plotly_white", title="Actual vs Forecasted Monthly Sales", hovermode="x unified")
    return fig


def backtest_chart(backtest_df: pd.DataFrame, model_name: str) -> go.Figure:
    """Compare actual and predicted values for the selected holdout model."""

    data = backtest_df.loc[backtest_df["Model"].eq(model_name)].copy()
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=data["Month"],
            y=data["Actual"],
            mode="lines+markers",
            name="Actual",
            line=dict(color="#111111", width=3),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=data["Month"],
            y=data["Predicted"],
            mode="lines+markers",
            name="Backtest prediction",
            line=dict(color="#E85D2A", width=3, dash="dash"),
        )
    )
    fig.update_layout(template="plotly_white", title="Holdout Backtest", hovermode="x unified")
    return fig


def cluster_diagnostic_chart(diagnostics: pd.DataFrame) -> go.Figure:
    """Show silhouette score for candidate customer cluster counts."""

    fig = px.line(
        diagnostics,
        x="Clusters",
        y="SilhouetteScore",
        markers=True,
        title="Cluster Count Diagnostic",
    )
    fig.update_traces(line_color="#167D6D", line_width=3)
    fig.update_layout(template="plotly_white", xaxis=dict(dtick=1))
    return fig


def product_action_scatter(product_df: pd.DataFrame) -> go.Figure:
    """Visualize product action categories by revenue and quantity."""

    fig = px.scatter(
        product_df,
        x="Quantity",
        y="Revenue",
        color="ActionCategory",
        hover_name="Description",
        log_x=True,
        log_y=True,
        title="Product Action Matrix",
        color_discrete_map={
            "Protect and promote": "#167D6D",
            "Premium-value opportunity": "#3659A2",
            "Volume / margin review": "#E9A23B",
            "Review or rationalize": "#D92D20",
        },
    )
    fig.update_layout(template="plotly_white")
    return fig


def quantity_histogram(clean_sales: pd.DataFrame) -> go.Figure:
    capped = clean_sales[clean_sales["Quantity"] <= clean_sales["Quantity"].quantile(0.99)]
    fig = px.histogram(capped, x="Quantity", nbins=50, title="Quantity Sold Distribution")
    fig.update_traces(marker_color="#111111")
    fig.update_layout(template="plotly_white")
    return fig

## 10. Run the Final Analytics Pipeline

Full-data preparation, KPI calculations, product analysis, cluster-count comparison, final three-cluster segmentation, revenue and quantity forecast comparison, and decision-support evidence.

In [20]:
pipeline_start = perf_counter()

raw_df = pd.read_excel(DATASET_PATH, engine="openpyxl")
prepared_all, clean_sales = prepare_retail_data(raw_df)
data_quality = summarize_data_quality(prepared_all)
kpis = calculate_overview_kpis(clean_sales)
monthly_df = monthly_sales(clean_sales)
country_df = sales_by_country(clean_sales, top_n=10)
products_df = product_performance(clean_sales, include_non_merchandise=False)
top_products_df = products_df.head(15)
product_actions_df = classify_product_actions(products_df)
customer_df = customer_value_table(clean_sales)
cancellations = cancellation_summary(prepared_all)

rfm_df = build_rfm_table(clean_sales)
cluster_diagnostics = evaluate_cluster_counts(rfm_df, minimum=2, maximum=6)
recommended_clusters = int(
    cluster_diagnostics.loc[cluster_diagnostics["SilhouetteScore"].idxmax(), "Clusters"]
)
segmentation_result = segment_customers(rfm_df, n_clusters=recommended_clusters)

revenue_forecast = run_forecasting(clean_sales, target="Revenue", periods=3, test_size=3)
quantity_forecast = run_forecasting(clean_sales, target="Quantity", periods=3, test_size=3)
insights = generate_business_insights(clean_sales, segmentation_result.rfm)

pipeline_seconds = perf_counter() - pipeline_start

print("Final pipeline completed.")
print("Raw rows:", f"{len(raw_df):,}")
print("Valid positive-sales rows:", f"{len(clean_sales):,}")
print("Recommended clusters:", recommended_clusters)
print("Silhouette score:", round(segmentation_result.metrics["silhouette_score"], 3))
print("Best revenue forecast model:", revenue_forecast.best_model)
print("Partial months excluded from forecasting:", revenue_forecast.excluded_partial_months)
print("Pipeline time:", round(pipeline_seconds, 3), "seconds")

Final pipeline completed.
Raw rows: 541,909
Valid positive-sales rows: 530,104
Recommended clusters: 3
Silhouette score: 0.416
Best revenue forecast model: Naive Last Value
Partial months excluded from forecasting: 1
Pipeline time: 81.933 seconds


## 11. Dataset Profile, Quality, and KPI Evidence

In [21]:
dataset_profile = {
    "rows": int(len(raw_df)),
    "columns": int(len(raw_df.columns)),
    "date_min": pd.to_datetime(raw_df["InvoiceDate"]).min(),
    "date_max": pd.to_datetime(raw_df["InvoiceDate"]).max(),
    "countries": int(raw_df["Country"].nunique()),
    "customers": int(raw_df["CustomerID"].nunique()),
    "orders": int(raw_df["InvoiceNo"].nunique()),
    "products": int(raw_df["StockCode"].nunique()),
}

display(pd.DataFrame([dataset_profile]))
display(pd.DataFrame([data_quality]))
display(pd.DataFrame([kpis]))
display(cancellations)

,rows,columns,date_min,date_max,countries,customers,orders,products
0,541909,8,2010-12-01 08:26:00,2011-12-09 12:50:00,38,4372,25900,4070


,total_rows,missing_customer_rows,cancelled_or_return_rows,zero_or_negative_price_rows,zero_or_negative_quantity_rows,valid_sales_rows,non_merchandise_sales_rows
0,541909,135080,10624,2517,10624,530104,2487


,total_revenue,total_orders,total_customers,average_order_value,total_quantity,unique_products,countries,date_min,date_max
0,1.066668e+07,19960,4338,534.403033,5588376.0,3922,38,2010-12-01 08:26:00,2011-12-09 12:50:00


{'cancelled_rows': 10624,
 'cancelled_invoices': 5172,
 'cancelled_quantity': -484531.0,
 'cancelled_value': -896812.49,
 'cancelled_share_rows': 0.019604767590130447}

## 12. Sales and Product Evidence

In [22]:
display(monthly_df)
display(country_df)
display(top_products_df[["StockCode", "Description", "Revenue", "Quantity", "Orders"]])
display(product_actions_df[["StockCode", "Description", "Revenue", "Quantity", "ActionCategory"]].head(20))

,Month,Revenue,Quantity,Orders,Customers,AverageOrderValue,IsCompleteMonth
0,2010-12-01,823746.140,359239,1559,885,528.381103,True
1,2011-01-01,691364.560,387785,1086,741,636.615617,True
2,2011-02-01,523631.890,283555,1100,758,476.028991,True
3,2011-03-01,717639.360,377526,1454,974,493.562146,True
4,2011-04-01,537808.621,308815,1246,856,431.628107,True
5,2011-05-01,770536.020,395738,1681,1056,458.379548,True
6,2011-06-01,761739.900,389213,1533,991,496.894912,True
7,2011-07-01,719221.191,401759,1475,949,487.607587,True
8,2011-08-01,759138.380,421770,1361,935,557.779853,True
9,2011-09-01,1058590.172,570820,1837,1266,576.260300,True


,Country,Revenue,Quantity,Orders,Customers
36,United Kingdom,9025222.084,4662390,18019,3920
24,Netherlands,285446.340,200361,94,9
10,EIRE,283453.960,147173,288,3
14,Germany,228867.140,119261,457,94
13,France,209715.110,112103,392,87
0,Australia,138521.310,83901,57,9
31,Spain,61577.110,27940,90,30
33,Switzerland,57089.900,30629,54,21
3,Belgium,41196.340,23237,98,25
32,Sweden,38378.330,36083,36,8


,StockCode,Description,Revenue,Quantity,Orders
1338,22423,REGENCY CAKESTAND 3 TIER,174484.74,13879,1988
2657,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1
3629,85123A,WHITE HANGING HEART T-LIGHT HOLDER,104340.29,37599,2189
2866,47566,PARTY BUNTING,99504.33,18295,1685
3608,85099B,JUMBO BAG RED RETROSPOT,94340.05,48474,2089
2114,23166,MEDIUM CERAMIC TOP STORAGE JAR,81700.92,78033,247
2022,23084,RABBIT NIGHT LIGHT,66964.99,30788,994
1021,22086,PAPER CHAIN KIT 50'S CHRISTMAS,64952.29,19355,1160
3405,84879,ASSORTED COLOUR BIRD ORNAMENT,59094.93,36461,1455
3048,79321,CHILLI LIGHTS,54117.76,10306,661


,StockCode,Description,Revenue,Quantity,ActionCategory
1338,22423,REGENCY CAKESTAND 3 TIER,174484.74,13879,Protect and promote
2657,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,Protect and promote
3629,85123A,WHITE HANGING HEART T-LIGHT HOLDER,104340.29,37599,Protect and promote
2866,47566,PARTY BUNTING,99504.33,18295,Protect and promote
3608,85099B,JUMBO BAG RED RETROSPOT,94340.05,48474,Protect and promote
2114,23166,MEDIUM CERAMIC TOP STORAGE JAR,81700.92,78033,Protect and promote
2022,23084,RABBIT NIGHT LIGHT,66964.99,30788,Protect and promote
1021,22086,PAPER CHAIN KIT 50'S CHRISTMAS,64952.29,19355,Protect and promote
3405,84879,ASSORTED COLOUR BIRD ORNAMENT,59094.93,36461,Protect and promote
3048,79321,CHILLI LIGHTS,54117.76,10306,Protect and promote


## 13. Customer Segmentation Evidence

In [23]:
display(cluster_diagnostics)
display(segmentation_result.summary)
display(segmentation_result.rfm[[
    "CustomerID", "Recency", "Frequency", "Monetary", "RFMScore", "SegmentName"
]].head(20))

,Clusters,SilhouetteScore,Inertia
0,2,0.406384,6872.736097
1,3,0.415666,4298.655783
2,4,0.379488,3242.098156
3,5,0.343854,2761.001195
4,6,0.332011,2397.012065


,Cluster,Customers,Recency,Frequency,Monetary,TotalRevenue,AvgOrderValue,AvgRFMScore,SegmentName
0,0,1326,30.199849,9.817496,5484.540317,7272500.460,633.766152,13.227753,Champions
2,2,2030,54.649754,2.041872,614.115189,1246653.833,338.508622,8.243350,Loyal Customers
1,1,982,255.029532,1.394094,399.443596,392253.611,296.127233,4.855397,At Risk


,CustomerID,Recency,Frequency,Monetary,RFMScore,SegmentName
1689,14646,2,73,280206.02,15,Champions
4201,18102,1,60,259657.30,15,Champions
3728,17450,8,46,194550.79,15,Champions
3008,16446,1,2,168472.50,13,Champions
1879,14911,1,201,143825.06,15,Champions
55,12415,24,21,124914.53,14,Champions
1333,14156,10,55,117379.63,15,Champions
3771,17511,3,31,91062.38,15,Champions
2702,16029,39,63,81024.84,13,Champions
0,12346,326,1,77183.60,7,Champions


## 14. Forecast Evaluation Evidence

In [24]:
print("Revenue forecast - best model:", revenue_forecast.best_model)
display(revenue_forecast.comparison)
display(revenue_forecast.backtest)
display(revenue_forecast.future_forecast)

print("Quantity forecast - best model:", quantity_forecast.best_model)
display(quantity_forecast.comparison)
display(quantity_forecast.future_forecast)

Revenue forecast - best model: Naive Last Value


,Model,MAE,RMSE,MAPE
0,Naive Last Value,481883.554000,519422.263291,37.423169
1,Moving Average (3 months),495168.207975,531091.721455,38.538530
2,Linear Trend,505060.551589,539372.138691,39.389216


,Month,Model,Actual,Predicted,AbsoluteError
0,2011-09-01,Naive Last Value,1058590.172,759138.380000,299451.792000
1,2011-10-01,Naive Last Value,1154979.300,759138.380000,395840.920000
2,2011-11-01,Naive Last Value,1509496.330,759138.380000,750357.950000
3,2011-09-01,Moving Average (3 months),1058590.172,746699.823667,311890.348333
4,2011-10-01,Moving Average (3 months),1154979.300,741686.464889,413292.835111
5,2011-11-01,Moving Average (3 months),1509496.330,749174.889519,760321.440481
6,2011-09-01,Linear Trend,1058590.172,730057.190194,328532.981806
7,2011-10-01,Linear Trend,1154979.300,735961.382411,419017.917589
8,2011-11-01,Linear Trend,1509496.330,741865.574628,767630.755372


,Month,Forecast
0,2012-01-01,1509496.33
1,2012-02-01,1509496.33
2,2012-03-01,1509496.33


Quantity forecast - best model: Naive Last Value


,Model,MAE,RMSE,MAPE
0,Naive Last Value,227806.000000,240544.433754,34.185063
1,Linear Trend,227929.711111,238545.313739,34.327951
2,Moving Average (3 months),241154.395062,252375.209581,36.319574


,Month,Forecast
0,2012-01-01,754507.0
1,2012-02-01,754507.0
2,2012-03-01,754507.0


## 15. Visual Evidence

In [25]:
revenue_trend(monthly_df).show()
country_bar(country_df).show()
product_bar(top_products_df, title="Top Merchandise Products by Revenue").show()
cluster_diagnostic_chart(cluster_diagnostics).show()
rfm_segment_scatter(segmentation_result.rfm).show()
segment_bar(segmentation_result.summary).show()
backtest_chart(revenue_forecast.backtest, revenue_forecast.best_model).show()
forecast_chart(revenue_forecast.monthly, revenue_forecast.future_forecast).show()

## 16. Decision-Support Evidence

In [26]:
for number, item in enumerate(insights, start=1):
    print(f"{number}. [{item['severity']}] {item['title']}")
    print(f"   Category: {item['category']}")
    print(f"   Evidence: {item['message']}")
    print(f"   Recommended focus: {item['recommendation']}\n")

1. [Positive] Recent revenue growth detected
   Category: Sales
   Evidence: Latest monthly revenue is 30.7% higher than the previous month.
   Recommended focus: Identify the products and countries driving the increase and protect stock availability.

2. [Positive] Top product deserves priority
   Category: Product
   Evidence: REGENCY CAKESTAND 3 TIER is the highest-revenue product in the dataset.
   Recommended focus: Prioritize availability, monitor demand, and consider promotion around this product category.

3. [Medium] Slow-moving product review needed
   Category: Product
   Evidence: GIRLY PINK TOOL SET has low revenue and quantity movement.
   Recommended focus: Review whether this item needs bundling, discounting, repositioning, or removal.

4. [Positive] High-value customer segment identified
   Category: Customer
   Evidence: The Champions segment contributes the largest customer revenue share.
   Recommended focus: Use loyalty, retention, or targeted communication strateg

## 17. Final Validation Tests


In [27]:
validation_checks = [
    ("Raw dataset row count", len(raw_df) == 541_909),
    ("Valid positive-sales row count", len(clean_sales) == 530_104),
    ("Full-data revenue", round(kpis["total_revenue"], 2) == 10_666_684.54),
    ("Full-data orders and customers", kpis["total_orders"] == 19_960 and kpis["total_customers"] == 4_338),
    ("Partial December 2011 identified", int((~monthly_df["IsCompleteMonth"]).sum()) == 1),
    ("Operational product codes excluded", "DOT" not in set(top_products_df["StockCode"].astype(str))),
    ("Cluster review recommends three segments", recommended_clusters == 3),
    ("Revenue forecast uses complete months", len(revenue_forecast.model_monthly) == 12 and revenue_forecast.excluded_partial_months == 1),
]

for name, passed in validation_checks:
    print(f"{'PASS' if passed else 'FAIL'} - {name}")

assert all(passed for _, passed in validation_checks), "At least one final validation check failed."
print(f"\nFinal result: {len(validation_checks)} of {len(validation_checks)} checks passed.")

PASS - Raw dataset row count
PASS - Valid positive-sales row count
PASS - Full-data revenue
PASS - Full-data orders and customers
PASS - Partial December 2011 identified
PASS - Operational product codes excluded
PASS - Cluster review recommends three segments
PASS - Revenue forecast uses complete months

Final result: 8 of 8 checks passed.


## 18. Local/Colab Performance Evidence

In [28]:
timings = {}

start = perf_counter(); _ = calculate_overview_kpis(clean_sales); timings["overview_kpis"] = perf_counter() - start
start = perf_counter(); _rfm = build_rfm_table(clean_sales); timings["rfm_table"] = perf_counter() - start
start = perf_counter(); _ = segment_customers(_rfm, n_clusters=3); timings["segmentation_k3"] = perf_counter() - start
start = perf_counter(); _ = run_forecasting(clean_sales, target="Revenue", periods=3, test_size=3); timings["forecast_comparison"] = perf_counter() - start

performance_df = pd.DataFrame([
    {"Operation": name, "Seconds": round(seconds, 3)} for name, seconds in timings.items()
])
display(performance_df)
print("These are environment-specific single-run measurements, not a cloud service guarantee.")

,Operation,Seconds
0,overview_kpis,0.367
1,rfm_table,0.782
2,segmentation_k3,1.171
3,forecast_comparison,0.565


These are environment-specific single-run measurements, not a cloud service guarantee.


## 19. Save Processed Data, Models, and Research Evidence


In [29]:
save_prepared_data(prepared_all, PROCESSED_DIR / "prepared_online_retail.csv.gz")
segmentation_result.rfm.to_csv(MODELS_DIR / "rfm_segments.csv", index=False)
segmentation_result.summary.to_csv(MODELS_DIR / "segment_summary.csv", index=False)
joblib.dump(
    {
        "model": segmentation_result.model,
        "scaler": segmentation_result.scaler,
        "metrics": segmentation_result.metrics,
        "recommended_clusters": recommended_clusters,
    },
    MODELS_DIR / "customer_segmentation.joblib",
)
joblib.dump(
    {
        "revenue": {
            "comparison": revenue_forecast.comparison,
            "backtest": revenue_forecast.backtest,
            "future_forecast": revenue_forecast.future_forecast,
            "best_model": revenue_forecast.best_model,
            "excluded_partial_months": revenue_forecast.excluded_partial_months,
        },
        "quantity": {
            "comparison": quantity_forecast.comparison,
            "backtest": quantity_forecast.backtest,
            "future_forecast": quantity_forecast.future_forecast,
            "best_model": quantity_forecast.best_model,
            "excluded_partial_months": quantity_forecast.excluded_partial_months,
        },
    },
    MODELS_DIR / "forecast_results.joblib",
)

cluster_diagnostics.to_csv(EVIDENCE_DIR / "cluster_diagnostics.csv", index=False)
segmentation_result.summary.to_csv(EVIDENCE_DIR / "segment_summary.csv", index=False)
revenue_forecast.comparison.to_csv(EVIDENCE_DIR / "forecast_comparison_revenue.csv", index=False)
quantity_forecast.comparison.to_csv(EVIDENCE_DIR / "forecast_comparison_quantity.csv", index=False)
top_products_df.to_csv(EVIDENCE_DIR / "top_merchandise_products.csv", index=False)
pd.DataFrame(insights).to_csv(EVIDENCE_DIR / "decision_support_insights.csv", index=False)
performance_df.to_csv(EVIDENCE_DIR / "performance_results.csv", index=False)


def json_value(value):
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.to_datetime(value).isoformat()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if pd.isna(value):
        return None
    return value


result_summary = {
    "dataset_profile": {key: json_value(value) for key, value in dataset_profile.items()},
    "data_quality": data_quality,
    "kpis": {key: json_value(value) for key, value in kpis.items()},
    "recommended_clusters": recommended_clusters,
    "segmentation_metrics": segmentation_result.metrics,
    "best_revenue_forecast_model": revenue_forecast.best_model,
    "revenue_forecast_metrics": revenue_forecast.comparison.to_dict(orient="records"),
    "partial_months_excluded": revenue_forecast.excluded_partial_months,
    "validation_checks_passed": sum(passed for _, passed in validation_checks),
    "validation_checks_total": len(validation_checks),
}
(EVIDENCE_DIR / "final_result_summary.json").write_text(
    json.dumps(result_summary, indent=2, default=json_value), encoding="utf-8"
)

print("Prepared deployment data:", PROCESSED_DIR / "prepared_online_retail.csv.gz")
print("Model artefacts:", MODELS_DIR)
print("Research evidence:", EVIDENCE_DIR)

Prepared deployment data: /content/data/processed/prepared_online_retail.csv.gz
Model artefacts: /content/models
Research evidence: /content/evidence


## 20. Download the Final Colab Outputs




In [30]:
archive_path = WORK_DIR / "BI_Dashboard_Final_Colab_Outputs.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in [PROCESSED_DIR, MODELS_DIR, EVIDENCE_DIR]:
        for file_path in folder.rglob("*"):
            if file_path.is_file():
                archive.write(file_path, file_path.relative_to(WORK_DIR))

print("Created:", archive_path)

try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print("Local run detected. The ZIP is available at the path above.")

Created: /content/BI_Dashboard_Final_Colab_Outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>